Đây là script chạy phân tích dữ liệu trước khi đi vào xử lý và huấn luyện mô hình. Chi tiết về việc phân tích và nội dung quan sát rút ra đều được ghi chi tiết trong notebook.

**Lưu ý**:
- Script **không bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Thời gian chạy cụ thể: 15-20 phút
- Với những trường hợp có thời gian chạy quá lớn (>30 phút), chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle.
- Cell code 3 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)

In [1]:
from collections import Counter
import numpy as np
import pandas as pd
import scipy.stats as ss
from scipy.stats import entropy

import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [2]:
INPUT_ROOT = "YOUR INPUT ROOT PATH"

X_train = pd.read_csv(f"{INPUT_ROOT}/X_train.csv")
X_val = pd.read_csv(f"{INPUT_ROOT}/X_val.csv")
X_test = pd.read_csv(f"{INPUT_ROOT}/X_test.csv")
Y_train = pd.read_csv(f"{INPUT_ROOT}/Y_train.csv")
Y_val = pd.read_csv(f"{INPUT_ROOT}/Y_val.csv")

In [ ]:
def get_unique_codes(df):
    features = [col for col in df.columns if col.startswith("feature_")]
    all_codes = df[features].values.flatten()
    valid_codes = all_codes[~np.isnan(all_codes)]
    return set(valid_codes)

train_codes = get_unique_codes(X_train)
val_codes = get_unique_codes(X_val)
test_codes = get_unique_codes(X_test)

train_val_intersect = train_codes.intersection(val_codes)
val_only = val_codes - train_codes                    
train_not_in_val = train_codes - val_codes             

train_test_intersect = train_codes.intersection(test_codes)
test_only = test_codes - train_codes

print(f"- Tổng số mã trong train: {len(train_codes)}")
print(f"- Tổng số mã trong val  : {len(val_codes)}")
print(f"- Tổng số mã trong test: {len(test_codes)}")
print(f"- Số mã có ở cả train và val: {len(train_val_intersect)}")
print(f"- Số mã có ở val nhưng không có ở train: {len(val_only)}")
print(f"- Số mã có ở train nhưng không có ở val: {len(train_not_in_val)}")
print(f"- Số mã có ở cả train và test: {len(train_test_intersect)}")
print(f"- Số mã có ở test nhưng không có ở train: {len(test_only)}")

print("\n--- KẾT LUẬN ---")
all_new_codes = (val_codes | test_codes) - train_codes
print(f"Tổng cộng có {len(all_new_codes)} mã mới sẽ xuất hiện lúc dự đoán.")

**1. Phân tích các mã từ X_train**

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 12})
plt.figure(figsize=(12, 6))

features = [col for col in X_train.columns if col.startswith("feature_")]
seq_lengths = X_train[features].notna().sum(axis=1)

ax1 = sns.histplot(seq_lengths, bins=range(1, 68), kde=True, color="royalblue")
ax1.set_title("Phân bố độ dài chuỗi hành động của khách hàng (Sequence Lengths)", fontsize=16,  pad=15)
ax1.set_xlabel("Số lượng hành động trong một phiên (Bước)", fontsize=14)
ax1.set_ylabel("Số lượng khách hàng", fontsize=14)

plt.tight_layout()
plt.show()

Biểu đồ phân phối lệch phải rõ rệt. Đa số khách hàng chỉ thao tác trong khoảng 10-20 bước, tuy nhiên tồn tại một phần nhỏ khách hàng kéo dài tới hơn 60 bước. Điều này khẳng định chuỗi hành vi của bài toán này có sự phân hóa độ dài rất lớn (Variable-length sequences), đòi hỏi mô hình phải có cơ chế Padding hoặc Attention linh hoạt.

In [ ]:
print("--- ĐỘ DÀI SEQUENCE (TRAIN)---")
print(seq_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999]))

print("\n--- CÁC MÃ HÀNH ĐỘNG ---")
all_actions = X_train[features].values.flatten()
all_actions = all_actions[~np.isnan(all_actions)]
action_counts = pd.Series(all_actions).value_counts()
print("\nTop 10 mã xuất hiện nhiều nhất:")
print(action_counts.head(10))

**Các nhóm hành vi**

- Nhóm hành động thường xuất hiện đầu
- Nhóm hành động xuất hiện cuối
- Nhóm hành động xuất hiện nhiều, nhưng vị trí không thuộc nhóm những mã xuất hiện đầu và cuối chuỗi 


- Phân tích tần suất xuất hiện của các mã, ta thấy 3 mã phổ biến nhất là mã 105, 102 và 103. Tuy nhiên, các mã này có xác suất xuất hiện ở đầu/cuối chuỗi thấp, nên những mã này sẽ là những mã đại diện cho các hành động phổ biến liên quan đến việc lướt ứng dụng/trình duyệt (browsing), xem sản phẩm, thêm vào giỏ hàng.

In [ ]:
sequences = []
for index, row in X_train.drop("id", axis=1).iterrows():
    seq = [int(x) for x in row.values if not np.isnan(x)]
    if seq:
        sequences.append(seq)


first_events = [seq[0] for seq in sequences]
last_events = [seq[-1] for seq in sequences]

print("Các sự kiện mở đầu:")
print(pd.Series(first_events).value_counts().head(5))

print("\nCác sự kiện kết thúc:")
print(pd.Series(last_events).value_counts().head(5))

- Nhóm hành động thường xuất hiện đầu: Các mã 760, 685, 836 thường đứng ở vị trí đầu tiên của chuỗi hành vi. Đây là các hành động "Mở ứng dụng" hoặc "Truy cập trang chủ". Việc có 2-3 mã cùng xuất hiện nhiều ở đầu chuỗi (Với 2 mã 760 cà 685 phổ biến hơn hẳn) cho thấy có thể đây là các mã đại diện cho việc truy cập từ nhiều nguồn khác nhau (Truy cập trực tiếp qua app, click từ link quảng cáo, ...).

- Nhóm hành động xuất hiện cuối: 1027, 995, 1080, 975 thường là các hành động cuối cùng chốt lại chuỗi. Đây có thể là các hành vi liên quan đến thanh toán thành công, huỷ đơn hàng, thoát ra khỏi application.

In [ ]:
view_codes = {105, 102, 103}
id_candidates = []
for s in sequences:
    for i in range(1, len(s) - 1):
        if s[i-1] in view_codes and s[i+1] in view_codes:
            id_candidates.append(s[i])
id_gr = [a for a, c in Counter(id_candidates).most_common(10)]

print("Nhóm ID của sản phẩm/ngành hàng:", id_gr[:10])

- Nhóm các mã có thể là ID của sản phẩm hoặc ID của danh mục sản phẩm: Các mã này thường có giá trị gần 1000 đi kèm (kẹp giữa) các mã nhóm điều hướng trung gian/xem sản phẩm (102,103,105). Ví dụ: Người dùng thực hiện thao tác xem, hệ thống ghi nhận họ đang xem sản phẩm 929, sau đó họ lại tiếp tục thao tác xem.

In [ ]:
all_actions = [a for s in sequences for a in s]
counts = Counter(all_actions)
transitions = []
for s in sequences:
    for i in range(len(s) - 1):
        transitions.append((s[i], s[i+1]))

trans_df = pd.DataFrame(transitions, columns=["From", "To"])

top_15 = [a for a, c in counts.most_common(15)]
matrix = pd.crosstab(trans_df["From"], trans_df["To"], normalize="index")
matrix_filtered = matrix.loc[matrix.index.isin(top_15), matrix.columns.isin(top_15)]

plt.figure(figsize=(12, 10))
sns.heatmap(matrix_filtered, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Ma trận xác suất chuyển đổi hành vi")
plt.xlabel("Hành động tiếp theo (B)")
plt.ylabel("Hành động hiện tại (A)")
plt.show()

- Sau bước đệm chung (mã 8615), người dùng có xu hướng tỏa ra nhiều hướng khác nhau. Chỉ khoảng 32% chuyển sang mã 105 (giả định là mã thể hiện việc người dùng sẽ lướt tiếp), số còn lại phân tán sang các thao tác khác, cho thấy đây là giai đoạn người dùng bắt đầu đưa ra quyết định cá nhân thay vì đi theo luồng mặc định.

In [ ]:
rare_threshold = 5
rare_codes = {code: count for code, count in counts.items() if count < rare_threshold}

print(f"Số lượng mã hiếm: {len(rare_codes)}")
print(f"Một số mã hiếm và số lần xuất hiện: {list(rare_codes.items())[:5]}")

In [ ]:
short_seqs = [seq for seq in sequences if len(seq) == 3]
count_103_middle = sum(1 for seq in short_seqs if seq[1] == 103)

print(f"Có {len(short_seqs)} chuỗi chỉ dài đúng 3 hành động.")
print(f"Trong đó, có {count_103_middle} chuỗi có mã 103 nằm ngay chính giữa")
print("Ví dụ 5 chuỗi đầu tiên:", short_seqs[:5])

- Đây là một nhóm người dùng (hoặc loại giao dịch) mang đặc thù riêng biệt. Rất có thể hành vi Mã A -> 103 -> Mã B đại diện cho một thao tác thanh toán tự động định kỳ, mua hàng nhanh (1-click purchase), hoặc hành vi của tài khoản mới tạo.

In [ ]:
bigrams = transitions
trigrams = []
for s in sequences:
    for i in range(len(s) - 2):
        trigrams.append(tuple(s[i:i+3]))

top_bi = Counter(bigrams).most_common(10)
top_tri = Counter(trigrams).most_common(10)

print(" Top 10 pattern 2 hành động liên tiếp:")
for p, c in top_bi: print(f"Pattern {p}: {c} lần")

print("\nTop 10 pattern 3 hành động liên tiếp:")
for p, c in top_tri: print(f"Pattern {p}: {c} lần")

Việc soi chiếu các hành động cụm 2 và 3 hành động phổ biến nhất cho thấy các flow lý tưởng của ứng dụng:

- Hai cụm hành vi phổ biến nhất là (685 -> 2207 -> 8615) và (760 -> 1593 -> 8615). Mặc dù bắt đầu từ các điểm vào khác nhau (685 và 760), tất cả đều hội tụ về mã 8615 trước khi người dùng thực hiện các hành vi mua sắm thực sự.
- Mã 8615 là ngưỡng khám phá của ứng dụng. Trước 8615, khách hàng đang ở trạng thái vào app, load trang. Sau 8615, họ mới thực sự bước vào trạng thái tâm lý mua sắm (Xuất hiện mã 105 ngay sau 8615).
- Nếu muốn triển khai các chương trình marketing hoặc hệ thống gợi ý cho các sản phẩm, thời điểm ngay sau mã hành động 8615 sẽ là thời điểm vàng vì đây là lúc ý định bắt đầu hình thành rõ nét nhất

**2. Phân tích các mã từ Y_Train**

In [ ]:
attrs = ["attr_1", "attr_2", "attr_3", "attr_4", "attr_5", "attr_6"]

desc_stats = Y_train[attrs].agg(["min", "max", "mean", "nunique"]).T
print("Thống kê mô tả 6 thuộc tính:")
print(desc_stats)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, attr in enumerate(attrs):
    top_vals = Y_train[attr].value_counts().head(20).reset_index()
    top_vals.columns = [attr, "count"]
    
    sns.barplot(data=top_vals, x=attr, y="count", ax=axes[i], 
                hue=attr, palette="YlGnBu_r", legend=False)
    
    axes[i].set_title(f"Phân phối top 20 của {attr}", fontsize=14)
    axes[i].set_xlabel("Giá trị mã")
    axes[i].set_ylabel("Số lượng")

plt.tight_layout()
plt.show()

In [ ]:
def check_date_validity(df, month_col, day_col):
    invalid_mask = (
        ((df[month_col] == 2) & (df[day_col] > 29)) |
        ((df[month_col].isin([4, 6, 9, 11])) & (df[day_col] > 30))
    )
    return invalid_mask.sum()

errors_1_2 = check_date_validity(Y_train, "attr_1", "attr_2")
errors_4_5 = check_date_validity(Y_train, "attr_4", "attr_5")

print(f"\nSố lỗi logic nếu (attr_1, attr_2) là Tháng - Ngày: {errors_1_2}")
print(f"Số lỗi logic nếu (attr_4, attr_5) là Tháng - Ngày: {errors_4_5}")

In [ ]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

corr_matrix = pd.DataFrame(index=attrs, columns=attrs)
for a1 in attrs:
    for a2 in attrs:
        corr_matrix.loc[a1, a2] = cramers_v(Y_train[a1], Y_train[a2])

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.astype(float), annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Ma trận tương quan Cramer's V giữa các thuộc tính")
plt.show()

Cặp ngày/tháng (1-4, 2-5) có sự tương quan mạnh, chứng tỏ thời gian bắt đầu và kết thúc của một đơn hàng bám rất sát nhau.

Cặp công suất (3-6) có sự tương quan cực thấp. Điều này cho thấy 2 chỉ số này đại diện cho 2 khâu vận hành hoàn toàn độc lập trong nhà máy (ví dụ: Chế biến vs Đóng gói). Chúng ta không thể dùng nhánh dự đoán của attr_3 để suy diễn cho attr_6.

In [ ]:
warnings.filterwarnings("ignore")
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 14, "axes.labelsize": 12, 
    "figure.autolayout": True, "font.family": "sans-serif"
})
sns.set_style("whitegrid")


p90_attr3 = Y_train["attr_3"].quantile(0.90)
p90_attr6 = Y_train["attr_6"].quantile(0.90)

Y_train["is_extreme_attr3"] = Y_train["attr_3"] >= p90_attr3
Y_train["is_extreme_attr6"] = Y_train["attr_6"] >= p90_attr6

df_merged = pd.merge(X_train, Y_train[["id", "is_extreme_attr3", "is_extreme_attr6"]], on="id")
feature_cols = [col for col in X_train.columns if col.startswith("feature_")]

action_counts = pd.Series(df_merged[feature_cols].values.flatten()).value_counts()
valid_actions = action_counts[action_counts >= 100].index

def calculate_lift_scores(df, target_col):
    """
    Tính Lift Score: Xác suất xuất hiện mã hành động ở nhóm quá tải 
    chia cho xác suất xuất hiện ở nhóm Bình Thường.
    """
    def get_probs(subset):
        actions = subset[feature_cols].values.flatten()
        valid_actions_in_subset = actions[~np.isnan(actions)]
        return pd.Series(valid_actions_in_subset).value_counts(normalize=True)
    
    prob_normal = get_probs(df[~df[target_col]])
    prob_extreme = get_probs(df[df[target_col]])
    
    lift_df = pd.DataFrame({"P_Extreme": prob_extreme, "P_Normal": prob_normal}).fillna(0)
    lift_df = lift_df.loc[lift_df.index.isin(valid_actions)]
    
    epsilon = 1e-8
    lift_df["Lift_Score"] = lift_df["P_Extreme"] / (lift_df["P_Normal"] + epsilon)
    return lift_df.sort_values("Lift_Score", ascending=False)

lift_attr3 = calculate_lift_scores(df_merged, "is_extreme_attr3")
lift_attr6 = calculate_lift_scores(df_merged, "is_extreme_attr6")

top_attr3 = lift_attr3.head(11)
top_attr6 = lift_attr6.head(11)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("Các mã hành động gây quá tải nhà máy", 
             fontsize=18, y=1.05)

sns.barplot(x=top_attr3["Lift_Score"], y=top_attr3.index.astype(int).astype(str), 
            palette="Reds_r", ax=axes[0])
axes[0].set_title("Khâu Vận Hành A (attr_3)", fontsize=16,  pad=15)
axes[0].set_xlabel("Hệ số Nâng (Lift Score)", fontsize=12)
axes[0].set_ylabel("Mã Hành Động (Action ID)", fontsize=12)
axes[0].axvline(x=1.0, color="grey", linestyle="--", linewidth=2)

for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.2f}x", 
                     (p.get_width() + 0.1, p.get_y() + p.get_height() / 2.), 
                     ha="left", va="center", fontsize=11, color="darkred")

sns.barplot(x=top_attr6["Lift_Score"], y=top_attr6.index.astype(int).astype(str), 
            palette="Blues_r", ax=axes[1])
axes[1].set_title("Khâu Vận Hành B (attr_6)", fontsize=16,  pad=15)
axes[1].set_xlabel("Hệ số Nâng (Lift Score)", fontsize=12)
axes[1].set_ylabel("Mã Hành Động (Action ID)", fontsize=12)
axes[1].axvline(x=1.0, color="grey", linestyle="--", linewidth=2)

for p in axes[1].patches:
    axes[1].annotate(f"{p.get_width():.2f}x", 
                     (p.get_width() + 0.1, p.get_y() + p.get_height() / 2.), 
                     ha="left", va="center", fontsize=11, color="darkblue")

plt.tight_layout()
plt.savefig("module2_dual_capacity_triggers.png", dpi=300, bbox_inches="tight")
plt.show()

set_attr3 = set(top_attr3.index)
set_attr6 = set(top_attr6.index)
common_triggers = set_attr3.intersection(set_attr6)


11 mã này đại diện cho những hành vi kinh doanh mang tính chất vĩ mô:
- Các đơn hàng B2B khổng lồ: Khách hàng sỉ gom hàng toàn bộ các danh mục.
- Yêu cầu cấu hình tùy chỉnh cực khó: Đòi hỏi cả xưởng chế biến lẫn xưởng đóng gói phải thiết lập lại dây chuyền từ đầu.
- Hành vi huỷ/đổi lệnh phút chót: Xóa bỏ toàn bộ lệnh sản xuất cũ để thay bằng lệnh mới, làm xáo trộn cả 2 khâu.

In [ ]:
plt.rcParams.update({"font.size": 12, "figure.autolayout": True})
sns.set_style("whitegrid")

daily_capacity = Y_train.groupby("attr_2")[["attr_3", "attr_6"]].mean().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(x="attr_2", y="attr_3", data=daily_capacity, marker="o", color="crimson", linewidth=2.5, label="Khâu A (attr_3)")
sns.lineplot(x="attr_2", y="attr_6", data=daily_capacity, marker="s", color="royalblue", linewidth=2.5, label="Khâu B (attr_6)")

plt.title("Biến động công suất theo ngày trong tháng", fontsize=16, pad=15)
plt.xlabel("Ngày bắt đầu giao dịch (attr_2: từ mùng 1 đến 31)", fontsize=14)
plt.ylabel("Công suất trung bình", fontsize=14)
plt.xticks(np.arange(1, 32, step=1))
plt.legend(fontsize=12)

# Highlight vùng cuối tháng
plt.axvspan(25, 31, color="orange", alpha=0.15, label="Vùng Dồn đơn (Rush zone)")
plt.savefig("eda_seasonality.png", dpi=300)
plt.show()

Qua các phân tích vĩ mô về thời gian và khối lượng, ta thấy dữ liệu không hề có tính chu kỳ hay các quy luật Business thông thường. Các thuộc tính mục tiêu gần như phân bố đồng đều theo thời gian. Điều này khẳng định: Nguyên nhân dẫn đến sự biến động của công suất nhà máy (attr_3, attr_6) hoàn toàn bắt nguồn từ sự phức tạp trong chuỗi hành vi vi mô của người dùng.

**3. Phân tích phân phối input từ tập train, val, test**

In [ ]:
train_len = seq_lengths
val_len   = X_val.drop("id", axis=1).count(axis=1)
test_len  = X_test.drop("id", axis=1).count(axis=1)


def plot_distribution_shift(train_col, val_col, test_col, feature_name):
    plt.figure(figsize=(10, 6))

    kde_colors = sns.color_palette("YlGnBu",10)
    
    sns.kdeplot(train_col, fill=True, label="Train", color=kde_colors[9], alpha=0.3)
    sns.kdeplot(val_col, fill=True, label="Val", color=kde_colors[2], alpha=0.3)
    sns.kdeplot(test_col, fill=True, label="Test", color=kde_colors[5], alpha=0.3)
    
    plt.title(f"{feature_name}", fontsize=14)
    plt.ylabel("Mật độ phân phối", fontsize=12)
    plt.legend(loc="upper right")
    plt.grid(True, linestyle="--", alpha=0.6)
    
    plt.show()

plot_distribution_shift(train_len, val_len, test_len, "Phân phối độ dài Session")